# Modular Exp Example in native FlagQuantum

Complete local three-cell Notebook workflow: prepare uniform powers and x=1, modular exponentiation, synthesize/execute, seeded sampling and verify the modular relation.

Preserve modulus five, base four, both three-bit unsigned registers and power/x output order. Fresh x initialization has the same semantics as allocation followed by XOR one. Use the shared modular permutation; 1000 seeded local samples replace the cloud sampler and ordinary tuple assertions replace dataframe queries. No hardware or cloud claim.

Define the native quantum functions below, synthesize them locally, inspect the executed results, and compare the complete complex state with an independent Circuit.

See [the native Python entry](main.py).

## Native imports for main

In [ ]:
from collections import Counter
from random import Random
from typing import Annotated
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    allocate,
    hadamard_transform,
    modular_exp,
    qfunc,
)

In [ ]:
N = 5

In [ ]:
A = 4

## main

In [ ]:
@qfunc
def main(power: Output[Annotated[QNum, 3]], x: Output[Annotated[QNum, 3]]) -> None:
    x |= 1
    allocate(power)
    hadamard_transform(power)
    modular_exp(N, A, x, power)

## build program

In [ ]:
def build_program() -> synth.Program:
    return synth.synthesize(main)

## run example

In [ ]:
def run_example() -> None:
    """Execute, sample 1000 named values, and check the original modular relation."""
    program = build_program()
    probabilities = dict(
        program.probabilities(fq.run(program), registers=("power", "x"))
    )
    counts = Counter(
        Random(17).choices(
            list(probabilities), weights=list(probabilities.values()), k=1000
        )
    )
    assert all(x == pow(A, power, N) for power, x in probabilities)
    assert all(x == pow(A, power, N) for power, x in counts)
    print(dict(counts))

## Execute the complete native workflow

Run every cell in order. Any sampling is local and seeded; resource reports come from the actual native program. Read the reported approximation limits and optimization outcomes alongside the numerical results.

In [ ]:
run_example()

## Verify absolute amplitudes and clean workspace

The independent Circuit uses the same declared register order. The comparison checks every complex amplitude without global-phase alignment and verifies zero probability in hidden work qubits. This is a local numerical check.


In [ ]:
import torch
from flagquantum import synth
from examples.synthesis.compare_states import compare_states
from examples.synthesis.functions.arithmetic.modular_exp.modular_exp_example.comparisons.circuit import (
    reference_circuit,
)

compare_states(build_program(), reference_circuit(), ("power", "x"))
print("Independent full-state comparison: passed")
